# Detección de tiña corporis con procesamiento clásico de imágenes
**PSIB 2026 Q2 — Grupo 9 (Del Carril, Escobar, Uthurralt)**

Pregunta: ¿se puede distinguir tiña corporis (TC) de dermatitis de contacto / eczema ("Otras") usando
características de color y textura, sin entrenar un modelo?

Idea central: la TC suele tener un **borde activo** (anillo rojo y descamado) con el **centro más claro**;
las lesiones inflamatorias de "Otras" tienden a ser placas más homogéneas.

## Pipeline
| Etapa | Qué hace | Archivo |
|---|---|---|
| 1 | Enmascarar fondo blanco + RGB → CIELAB | `etapas/e1_mascara_lab.py` |
| 2 | Mejoramiento: mediana en a* y L*, ecualización de a* (con máscara) | `etapas/e2_mejoramiento.py` |
| 3 | Segmentación de la lesión: comparación de métodos y region growing con semillas | `etapas/e3_segmentacion.py` |
| 4 | Separación borde / centro de la lesión (morfología) | *pendiente* |
| 5 | Características: color (a*) y textura (GLCM) en borde vs. centro | *pendiente* |
| 6 | Decisión estadística: comparación entre clases y elección del umbral | *pendiente* |
| 7 | Evaluación en testeo | *pendiente* |

Las funciones de cada etapa están en `etapas/`; este notebook solo las llama y muestra resultados.

**Regla de trabajo:** las etapas 1–6 se desarrollan y ajustan **solo con Entrenamiento** (`CONJUNTO = "Entrenamiento"`). Testeo no se mira ni se usa para elegir nada; se carga una única vez en la sección 7, con todos los parámetros y umbrales ya fijados.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from etapas.datos import listar_dataset, verificar_separacion, cargar_rgb
from etapas import e1_mascara_lab as e1
from etapas import e2_mejoramiento as e2
from etapas import e3_segmentacion as e3

RUTA_DATASET = Path("Dataset_reducido")
CONJUNTO = "Entrenamiento"   # Testeo se usa solo en la sección 7
RUTA_RESULTADOS = Path("resultados")   # se crea sola; cada etapa guarda en resultados/etapaN/

## 0. Dataset

In [ ]:
verificar_separacion(RUTA_DATASET)   # ninguna imagen en ambos conjuntos

df = listar_dataset(RUTA_DATASET, CONJUNTO)
print(f"Conjunto: {CONJUNTO} — {len(df)} imágenes")
df.groupby(["clase", "subtipo"]).size().rename("n").to_frame()

## 1. Enmascarado del fondo y conversión a Lab

**Por qué primero:** varias imágenes tienen el fondo recortado en blanco. Si se dejan esos píxeles,
Otsu termina separando fondo vs. resto y la GLCM mide pares blanco-blanco y el salto piel/fondo en vez de la textura de la piel.

**Máscara:** píxeles claros y sin color (HSV: V ≥ 235, S ≤ 25) que forman una región **conectada al borde** de la imagen.
Ese último criterio es importante: la descamación blanquecina dentro de la lesión también es clara y poco saturada,
pero queda rodeada de piel y no se borra. Después se dilata el fondo unos píxeles para descartar la franja de transición.

**Lab:** separa la luminosidad (L*) del color. El eritema se mide en **a*** (eje verde–rojo) sin que influya
si la foto está más clara u oscura.

In [ ]:
resultados = {id_: e1.procesar_etapa1(cargar_rgb(r)) for id_, r in df["ruta"].items()}
df["frac_fondo"] = pd.Series({k: v["frac_fondo"] for k, v in resultados.items()})

print(f"Imágenes con fondo descartado: {(df['frac_fondo'] > 0).sum()} de {len(df)}")
(df.groupby("clase")["frac_fondo"]
   .agg(con_fondo=lambda s: (s > 0).sum(), n="size", fondo_medio="mean")
   .style.format({"fondo_medio": "{:.1%}"}))

### Guardado
Se guarda en `resultados/etapa1/<conjunto>/`, por imagen:
- `<id>_mascara.png`: máscara de piel (blanco = piel).
- `<id>_enmascarada.png`: imagen con el fondo en negro, para mirar.
- `<id>_lab.npz`: L*, a*, b* en float y la máscara. **Es lo que lee la etapa 2** (un PNG perdería los decimales y los valores negativos de a*/b*).

In [ ]:
carpeta_e1 = RUTA_RESULTADOS / "etapa1"
for id_, res in resultados.items():
    rutas = e1.guardar_etapa1(res, id_, carpeta_e1 / CONJUNTO)
    df.loc[id_, "ruta_lab"] = rutas["lab"]

print(f"Guardadas {len(resultados)} imágenes en {(carpeta_e1 / CONJUNTO).resolve()}")

### Ejemplos
- **CD (120)**: mucho fondo blanco recortado.
- **EC (86)**: tiene escamas blancas dentro de la lesión → no deben marcarse como fondo.
- **TC (156)**: anillo típico; en a* el borde debería verse más rojo que el centro.
- **TC (8)**: lesión clara con poco fondo en una esquina.

In [ ]:
for id_ in ["CD (120)", "EC (86)", "TC (156)", "TC (8)"]:
    e1.mostrar_etapa1(resultados[id_], titulo=f"{id_} — {df.loc[id_, 'clase']}")
    plt.show()

### Control de todas las máscaras
Fondo descartado en azul. Revisar que no se esté comiendo piel ni escamas.

In [ ]:
ids = df.sort_values("clase").index
cols = 8
filas = int(np.ceil(len(ids) / cols))
fig, axs = plt.subplots(filas, cols, figsize=(2.2 * cols, 2.4 * filas))
for ax, id_ in zip(axs.flat, ids):
    r = resultados[id_]
    img = r["rgb"].copy()
    img[~r["mascara"]] = (0, 120, 255)
    ax.imshow(img)
    ax.set_title(f"{id_} · {df.loc[id_, 'clase']}\nfondo {100 * r['frac_fondo']:.0f}%", fontsize=8)
for ax in axs.flat:
    ax.axis("off")
fig.tight_layout()
plt.show()

## 2. Mejoramiento

Cada canal se prepara según para qué se va a usar después:

| Canal | Cómo | Para qué |
|---|---|---|
| `a_suave` | mediana 7×7 sobre a* | segmentar y medir color (etapas 3 y 5) |
| `a_eq` | ecualización de `a_suave`, solo con píxeles de piel | **solo** segmentar / mirar |
| `L` | L* sin tocar | textura GLCM (etapa 5) |
| `L_suave` | mediana 7×7 sobre L* | segmentar (sin pelos ni pliegues finos) |

**Mediana.** Saca lo que es más fino que el kernel y respeta los bordes (el gaussiano los desdibuja).
- En **a\***, el pelo casi no aparece (es oscuro, no rojo). Lo que limpia la mediana es el patrón en bloques del JPEG,
  que guarda el color con la mitad de resolución que el brillo.
- En **L\***, la mediana sí borra pelos y pliegues finos, pero **también la descamación**, que es justo la textura que mide la GLCM.
  Por eso se guardan dos versiones de L*: suavizada para segmentar y original para textura.
- Se probó detectar pelos con black-hat (cierre − imagen) para excluirlos, pero también marca escamas y pliegues
  (24% de la imagen en CD 56, y el anillo entero en TC 156). Se descartó.

**¿Por qué ecualizar a\* y no L\*?**
- **a\*** es la variable que separa lesión de piel, pero en cada imagen la piel ocupa un rango muy angosto (~12 unidades de ±128 posibles).
  Ecualizar estira ese rango y la lesión resalta.
- En **L\***, las diferencias grandes no son "lesión vs. piel" sino iluminación, sombras y tono de piel. Ecualizar L\* resalta sombras y ruido,
  y cambia los niveles de gris que después mide la GLCM.
- **Cuidado:** la ecualización es por imagen (cada imagen se estira según su propio histograma). Un valor de `a_eq` no significa lo mismo
  en dos imágenes, así que **no se usa para medir características**, solo para segmentar.
- La ecualización se calcula **solo con los píxeles de piel**: si entra el fondo blanco (a* ≈ 0, hasta 36% de la imagen) se lleva parte del contraste.

In [ ]:
resultados_e2 = {id_: e2.procesar_etapa2(resultados[id_]) for id_ in df.index}

carpeta_e2 = RUTA_RESULTADOS / "etapa2" / CONJUNTO
for id_, r in resultados_e2.items():
    df.loc[id_, "ruta_e2"] = e2.guardar_etapa2(r, id_, carpeta_e2)
print(f"Guardadas {len(resultados_e2)} imágenes en {carpeta_e2.resolve()}")

### Ejemplos
- **TC (156)**: anillo con escamas. El anillo tiene que seguir visible después de la mediana.
- **TC (207)** y **EC (385)**: tienen pelos. Fijarse que en a\* casi no se ven, y que en `L_suave` desaparecen.
- **CD (120)**: mucho fondo blanco.

In [ ]:
for id_ in ["TC (156)", "TC (207)", "EC (385)", "CD (120)"]:
    e2.mostrar_etapa2(resultados[id_], resultados_e2[id_], titulo=f"{id_} — {df.loc[id_, 'clase']}")
    plt.show()

### Por qué la ecualización va con máscara
En CD (120), con 36% de fondo, el histograma de toda la imagen tiene un pico enorme del fondo (gris) que no es piel.
Con `cv2.equalizeHist` ese pico consume parte del rango de salida y la piel queda menos contrastada.

In [ ]:
id_ = "CD (120)"
e2.mostrar_efecto_mascara_en_ecualizacion(resultados[id_]["a"], resultados[id_]["mascara"], titulo=id_)
plt.show()

### Por qué no se ecualiza L\*
Misma ecualización con máscara, aplicada a L\*: resaltan las sombras, la curvatura del miembro y el ruido, y no la lesión.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for fila, id_ in enumerate(["TC (171)", "TC (207)"]):
    r1, r2 = resultados[id_], resultados_e2[id_]
    m = r2["mascara"]
    ax[fila, 0].imshow(r1["rgb"]); ax[fila, 0].set_title(id_)
    ax[fila, 1].imshow(np.where(m, r2["L"], np.nan), cmap="gray"); ax[fila, 1].set_title("L* original")
    ax[fila, 2].imshow(np.where(m, e2.ecualizar_con_mascara(r2["L"], m), np.nan), cmap="gray"); ax[fila, 2].set_title("L* ecualizado")
for a in ax.flat:
    a.axis("off")
fig.tight_layout()
plt.show()

## 3. Segmentación de la lesión

Objetivo: una máscara de la **lesión completa**, borde **y** centro, porque la etapa 4 compara uno con otro.

Todos los métodos trabajan sobre **a\*** (la lesión se define por ser más roja que la piel) y dentro de la máscara de piel.
Para compararlos en igualdad de condiciones, todos pasan por el mismo **posprocesamiento** (clase de Segmentación II):
1. **Apertura** (disco r=3): borra puntitos sueltos.
2. **Cierre** (disco r=15): une tramos cercanos del anillo.
3. **Relleno de huecos**: recupera el centro de la TC, que es casi del color de la piel.
4. **Etiquetado**: se descartan las componentes chicas (<20% de la mayor o <0,5% de la piel).

En cada figura, la fila de arriba es la salida **cruda** del método (verde) y la de abajo el resultado **final** (contorno).
Ejemplos: **TC (156)** y **TC (207)** (anillos), **EC (385)** (lesión roja de Otras) y **EC (37)** (placa blanca descamada, poco roja).

In [ ]:
EJEMPLOS_SEG = ["TC (156)", "TC (207)", "EC (385)", "EC (37)"]

### Por qué cuesta: el histograma de a\* tiene un solo pico
La piel sana forma un pico y la lesión es apenas una **cola** hacia la derecha. Otsu supone dos picos (clase: *"solo da buenos
resultados cuando el histograma es bimodal"*): con uno solo, corta el pico de la piel por la mitad. Lo mismo pasa con Li y Triangle.
El umbral relativo a la piel (rojo) cae en la cola.

In [ ]:
e3.mostrar_histogramas(["TC (156)", "TC (52)", "EC (385)"], resultados_e2)
plt.show()

### Otsu (a* suave)
**Otsu sobre a\* suavizado.** Elige el umbral que maximiza la varianza entre las dos clases. Supone histograma bimodal.

In [ ]:
e3.mostrar_metodo("Otsu (a* suave)", EJEMPLOS_SEG, resultados, resultados_e2)
plt.show()

### Otsu (a* ecualizado)
**Otsu sobre a\* ecualizado.** La ecualización estira el contraste, pero no crea un segundo pico: Otsu sigue partiendo la piel.

In [ ]:
e3.mostrar_metodo("Otsu (a* ecualizado)", EJEMPLOS_SEG, resultados, resultados_e2)
plt.show()

### Li
**Li.** Umbral de mínima entropía cruzada. Con histogramas de un pico se comporta parecido a Otsu.

In [ ]:
e3.mostrar_metodo("Li", EJEMPLOS_SEG, resultados, resultados_e2)
plt.show()

### Triangle
**Triangle.** Pensado para un pico grande y una cola, que es nuestro caso, pero el umbral cae muy cerca del pico y se lleva mucha piel.

In [ ]:
e3.mostrar_metodo("Triangle", EJEMPLOS_SEG, resultados, resultados_e2)
plt.show()

### Adaptativo
**Umbral adaptativo (local).** Cada píxel se compara con la media de su entorno (101×101 px). Marca bordes y detalles, pero en una lesión grande el interior también es "normal" para su entorno y se pierde.

In [ ]:
e3.mostrar_metodo("Adaptativo", EJEMPLOS_SEG, resultados, resultados_e2)
plt.show()

### k-means (a*, L*)
**k-means (k=2) sobre a\* y L\*.** Agrupa por color y brillo. Con dos grupos termina separando zonas claras de oscuras (iluminación) más que lesión de piel.

In [ ]:
e3.mostrar_metodo("k-means (a*, L*)", EJEMPLOS_SEG, resultados, resultados_e2)
plt.show()

### Contorno activo
**Contorno activo (Chan-Vese).** Ajusta una curva para que adentro y afuera sean homogéneos. Da contornos suaves, pero busca la misma partición en dos que Otsu, y es ~30 veces más lento.

In [ ]:
e3.mostrar_metodo("Contorno activo", EJEMPLOS_SEG, resultados, resultados_e2)
plt.show()

### Umbral piel + k·MAD
**Umbral relativo a la piel: mediana + 2,5·MAD.** La mediana de a\* representa la piel sana (la lesión suele ser minoría) y la MAD es su dispersión. Solo marca lo claramente más rojo que esa piel: casi sin falsos positivos, pero se queda con los puntos más rojos y pierde parte de la lesión.

In [ ]:
e3.mostrar_metodo("Umbral piel + k·MAD", EJEMPLOS_SEG, resultados, resultados_e2)
plt.show()

### Region growing (semillas)
**Crecimiento de región con semillas.** Semillas: píxeles con a\* > mediana + 2,5·MAD (claramente rojos). Crecimiento: se suman los vecinos conectados con a\* > mediana + 1,5·MAD. Una zona moderadamente roja entra solo si está pegada a una semilla, y así se recupera la lesión sin traer piel suelta.

In [ ]:
e3.mostrar_metodo("Region growing (semillas)", EJEMPLOS_SEG, resultados, resultados_e2)
plt.show()

### Comparación sobre todo Entrenamiento
No hay segmentaciones manuales de referencia, así que la comparación combina lo que se ve en las figuras con dos indicadores simples:
- **% de piel marcada como lesión**: en estas fotos la lesión rara vez ocupa más de la mitad de la piel. Un método que marca ~50% en casi todas está partiendo la piel, no encontrando lesiones.
- **Contraste de a\***: (mediana adentro − mediana afuera) / dispersión. Cuanto más alto, más roja es la región marcada respecto del resto.

(El contorno activo tarda unos minutos.)

In [ ]:
filas = []
for nombre in e3.METODOS:
    for id_ in df.index:
        _, final = e3.segmentar(resultados_e2[id_], nombre)
        filas.append({"método": nombre, "id": id_, **e3.describir(final, resultados_e2[id_])})
comparacion = pd.DataFrame(filas)
(comparacion.groupby("método")
    .agg(pct_piel_mediana=("frac_lesion", "median"),
         imgs_mas_50pct=("frac_lesion", lambda x: (x > 0.5).sum()),
         imgs_sin_lesion=("frac_lesion", lambda x: (x == 0).sum()),
         contraste_a_mediana=("contraste_a", "median"))
    .sort_values("contraste_a_mediana", ascending=False)
    .style.format({"pct_piel_mediana": "{:.0%}", "contraste_a_mediana": "{:.2f}"}))

### Conclusión
**Método elegido: crecimiento de región con semillas relativas a la piel** (`e3.METODO_ELEGIDO`).

- **Los métodos globales no sirven acá** (Otsu, Li, Triangle, Otsu ecualizado, k-means, Chan-Vese). Todos asumen dos poblaciones de tamaño parecido,
  pero el histograma de a\* tiene un solo pico. Terminan marcando cerca de la mitad de la piel y siguen la iluminación y las sombras, no la lesión.
- **El adaptativo** encuentra bordes y manchas chicas, pero pierde el interior de las lesiones grandes.
- **El umbral relativo a la piel** es el único criterio que se adapta a cada imagen (tono de piel, iluminación) sin suponer bimodalidad,
  pero solo con el umbral alto se queda con los puntos más rojos.
- **El crecimiento de región** usa ese umbral alto como semilla y crece con uno más bajo. Así recupera la lesión sin traer piel suelta.
  Es el de mayor contraste de a\* con un % de piel razonable.

**Limitaciones** (se ven en la grilla de abajo):
- Las lesiones **poco rojas** (placas blancas descamadas, manchas oscuras) no se detectan bien, porque el criterio es el eritema.
- En varias TC el anillo es apenas más rojo que la piel y se recupera **solo en parte**. Si el anillo queda abierto, el relleno de huecos no recupera el centro.
- Si la lesión ocupa la mayor parte de la foto, la mediana deja de representar a la piel sana.

### Aplicación a todas las imágenes de Entrenamiento

In [ ]:
lesiones = {id_: e3.segmentar(resultados_e2[id_])[1] for id_ in df.index}
df["frac_lesion"] = pd.Series({id_: lesiones[id_][resultados_e2[id_]["mascara"]].mean() for id_ in df.index})

carpeta_e3 = RUTA_RESULTADOS / "etapa3" / CONJUNTO
for id_, les in lesiones.items():
    df.loc[id_, "ruta_lesion"] = e3.guardar_etapa3(les, id_, carpeta_e3)
print(f"Método: {e3.METODO_ELEGIDO}. Guardadas {len(lesiones)} máscaras en {carpeta_e3.resolve()}")
print(f"Imágenes sin lesión detectada: {(df['frac_lesion'] == 0).sum()}")

In [ ]:
ids = df.sort_values(["clase", "subtipo"]).index
e3.grilla_segmentaciones(ids, lesiones, resultados, resultados_e2,
                         etiquetas=df["clase"].to_dict(), cols=6)
plt.show()

### Resultado en Entrenamiento (inspección visual de la grilla de arriba)

| | Todas (37) | Solo TC (13) |
|---|---|---|
| Lesión bien segmentada | 7 | 1 — TC (207) |
| Parcial (parte de la lesión, p. ej. medio anillo) | 7 | 5 — TC (156), (208), (11), (171), (203) |
| Marca piel sana o bordes | 13 | 6 — TC (139), (27), (52), (56), (8), (206) |
| No detecta nada | 10 | 1 — TC (91) |

Funciona en lesiones **rojas y bien delimitadas** (EC 385, EC 86, EC 381, CD 45, TC 207). En la mayoría de las TC el anillo es apenas
más rojo que la piel y queda **abierto**, así que el relleno de huecos no recupera el centro. Con esta segmentación automática,
la comparación borde/centro de la etapa 4 no es confiable en la mayoría de las imágenes: ver la decisión pendiente en el resumen.

## 4. Borde vs. centro
*Pendiente.*

## 5. Características
*Pendiente.*

## 6. Decisión estadística
*Pendiente.*

## 7. Evaluación en testeo
*Pendiente.* Se corre una sola vez al final: `listar_dataset(RUTA_DATASET, "Testeo")`, mismas funciones y mismos parámetros/umbrales fijados en las etapas 1–6, sin volver a ajustarlos según el resultado.